# LLM auto-grading -- Kaggle runner

Clones the public code repo ([https://github.com/ptran1203/msc.git](https://github.com/ptran1203/msc.git)) and attaches the private data
via the Kaggle dataset `ptran1203/llm-autograding-course-data`, plus the synthetic bug-injection data
via the `ptran1203/llm-synth-data-generator` notebook's output (both added automatically when
pushed via `kernel_sources`/`dataset_sources` in kernel-metadata.json). Course rules forbid
distributing the real student data outside the class, so it lives only in that private
dataset, never in the repo itself.

Edit the **config cell** below, then Run All.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
%cd /kaggle/working
!rm -rf msc
!git clone --depth 1 https://github.com/ptran1203/msc.git
%cd /kaggle/working/msc/llm
!pip install -q -r requirements.txt


In [ ]:
import shutil, pathlib
dataset_root = pathlib.Path("/kaggle/input/datasets/ptran1203/llm-autograding-course-data")
synth_root = pathlib.Path("/kaggle/input/notebooks/ptran1203/llm-synth-data-generator/synth_generated")
repo_llm = pathlib.Path("/kaggle/working/msc/llm")
shutil.copytree(dataset_root / "data", repo_llm / "data")
(repo_llm / "synth").mkdir(exist_ok=True)
shutil.copytree(synth_root, repo_llm / "synth" / "generated")
print("data staged OK")


## Run -- generates the public submission (328 samples, all 3 tasks), TWO configs

Runs **both** `--fewshot` (static hand-picked real exemplars) and `--rag` (retrieval from
the real + synthetic pool) back to back, so the two can be compared directly on the
leaderboard instead of guessing from one run. Both now also get the **real compile_log**
(see `compile_check.py`): the released data leaves it empty, so the model previously had
to guess compilability purely by reading code and was wrong ~20-25% of the time even on
code that compiles -- confirmed the actual cause of Task 1's negative QWK, not RAG vs.
few-shot (which scored almost identically, -0.2171 vs -0.2108, before this fix).

`--rag` had anchored Task 2 hard toward single-label outputs last time (every synthetic
exemplar has exactly one injected bug), so it's still a real risk for that task
specifically -- that's exactly what running both configs side by side is for.

Each config runs across **both GPUs in parallel** (`--num-shards 2`, one process per GPU
via `CUDA_VISIBLE_DEVICES`, merged by `merge_shards.py`). **Needs the kernel's accelerator
set to "GPU T4 x2"** in the Kaggle notebook editor (Settings -> Accelerator) -- not
exposed through the push API, so it has to be set once by hand and then persists across
future `kaggle kernels push` updates.

In [ ]:
%cd /kaggle/working/msc/llm/src
import os, subprocess

RUN_TAGS = {"fewshot": "qwen-fewshot", "rag": "qwen-rag"}
METHOD_FLAGS = {"fewshot": "--fewshot", "rag": "--rag"}

for method, run_tag in RUN_TAGS.items():
    cmd = lambda shard: [
        "python", "-u", "run_prompting.py", "--split", "public", "--tasks", "1,2,3",
        METHOD_FLAGS[method], "--run-tag", run_tag, "--num-shards", "2", "--shard-id", str(shard),
    ]
    envs = [{**os.environ, "CUDA_VISIBLE_DEVICES": str(i)} for i in range(2)]
    procs = [subprocess.Popen(cmd(i), env=envs[i]) for i in range(2)]
    for p in procs:
        p.wait()
    assert all(p.returncode == 0 for p in procs), (method, [p.returncode for p in procs])
    subprocess.run(
        ["python", "merge_shards.py", "--split", "public", "--tasks", "1,2,3",
         "--method-tag", run_tag, "--num-shards", "2"],
        check=True,
    )
    print(f"=== {method} done ===")


In [ ]:
import glob, re
for path in sorted(glob.glob("../outputs/predictions_task*_public_*.json")):
    if "_shard" in path:
        continue  # per-GPU partial outputs, not the merged submission files
    task_num = re.search(r"predictions_task(\d)_", path).group(1)
    get_ipython().system(f"python validate_submission.py {task_num} {path} --split public")


## Collect submissions + clean up

Copies the 6 final (validated) predictions files -- 3 tasks x 2 configs -- into
`/kaggle/working/submission/` under plain `predictions_task{N}_{fewshot,rag}.json` names,
then deletes the cloned repo -- so the notebook's saved output is only that one folder
instead of the whole source tree + per-shard intermediates.

In [ ]:
import shutil, pathlib
sub_dir = pathlib.Path("/kaggle/working/submission")
sub_dir.mkdir(exist_ok=True)
for method, run_tag in RUN_TAGS.items():
    for t in (1, 2, 3):
        src = pathlib.Path(f"/kaggle/working/msc/llm/outputs/predictions_task{t}_public_{run_tag}.json")
        if src.exists():
            shutil.copy(src, sub_dir / f"predictions_task{t}_{method}.json")
%cd /kaggle/working
shutil.rmtree("/kaggle/working/msc", ignore_errors=True)
print("submission files:", sorted(p.name for p in sub_dir.iterdir()))


## Output

`/kaggle/working/submission/predictions_task{1,2,3}_{fewshot,rag}.json` -- pick whichever
config scores better per task directly, already validated against the schema above.